# Paper PoRT Post-Judge Routing Semantics Diagnostic

This notebook follows notebook 27. It keeps the same `32` rows per variant/domain job, but focuses on route semantics after initial and rethink answers already exist.

It compares current paper-style routing, inverted correctness routing, confidence-only routing, and row-level oracle upper bounds for raw and structure-gated prompts.

This is a recreated-artifact diagnostic, not an official paper-checkpoint metric run. Oracle methods use ground-truth correctness and are upper bounds only.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Key defaults:

- `PORT_MAX_SAMPLES=32`
- `PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT=true`
- `PORT_BOOTSTRAP_RECREATED_IF_MISSING=false`
- `PORT_BEST_CLASSIFIER_FEATURE_SET=answer_only`
- `PORT_CLASSIFIER_CONF_THRESHOLD=0.70`

Route policies reported for raw and structure-gated prompts:

- `paper_keep_label0_conf`: keep `label==0` only when confidence is high; rethink otherwise.
- `paper_keep_label0_no_conf`: keep `label==0`; rethink otherwise.
- `inverted_keep_label1_conf`: keep `label==1` only when confidence is high; rethink otherwise.
- `inverted_keep_label1_no_conf`: keep `label==1`; rethink otherwise.
- `confidence_keep_high`: keep high-confidence initial answers regardless of label.
- `confidence_rethink_high`: rethink high-confidence initial answers regardless of label.

The runner performs a CUDA preflight before loading the target model. If Kaggle raises `cudaErrorNoKernelImageForDevice`, switch to a supported GPU accelerator, preferably T4 as used by the prior successful runs.

In [ ]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_RUN_NAME', 'paper_port_wmdp_postjudge_routing_semantics_diagnostic_phi-1_5')
os.environ.setdefault('PORT_MAX_SAMPLES', '32')
os.environ.setdefault('PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT', 'true')
os.environ.setdefault('PORT_RECREATED_ARTIFACT_MANIFEST_URL', 'https://raw.githubusercontent.com/toanthangO20/PoRT_LLM_Unlearning-Experiment/artifact-recreated-bootstrap-v1/manifest.json')
os.environ.setdefault('PORT_BOOTSTRAP_RECREATED_IF_MISSING', 'false')
os.environ.setdefault('PORT_QUALITY_GATE_MIN_CHOICE_COVERAGE', '1.0')
os.environ.setdefault('PORT_QUALITY_GATE_MIN_LEN_RATIO', '0.50')
os.environ.setdefault('PORT_QUALITY_GATE_MAX_LEN_RATIO', '2.00')
os.environ.setdefault('PORT_QUALITY_GATE_REQUIRE_PROMPT_INSTRUCTION', 'true')
os.environ.setdefault('PORT_QUALITY_GATE_REQUIRE_ANSWER_INSTRUCTION', 'false')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT',
    'PORT_RECREATED_ARTIFACT_MANIFEST_URL',
    'PORT_BOOTSTRAP_RECREATED_IF_MISSING',
    'PORT_QUALITY_GATE_MIN_CHOICE_COVERAGE',
    'PORT_QUALITY_GATE_MIN_LEN_RATIO',
    'PORT_QUALITY_GATE_MAX_LEN_RATIO',
    'PORT_QUALITY_GATE_REQUIRE_PROMPT_INSTRUCTION',
    'PORT_QUALITY_GATE_REQUIRE_ANSWER_INSTRUCTION',
    'PORT_CLASSIFIER_CONF_THRESHOLD',
    'PORT_BEST_CLASSIFIER_FEATURE_SET',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
    'PORT_RECREATED_ARTIFACT_DIR',
    'PORT_RECREATED_ARTIFACT_ZIP_URL',
    'PORT_RECREATED_ARTIFACT_ZIP_PATH',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


In [ ]:
from pathlib import Path
import gc
import importlib.util
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


if 'PROJECT_ROOT' not in globals() or not has_project_layout(PROJECT_ROOT):
    PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if 'commit_sha' not in globals():
    commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('Cleared CUDA cache before run.')
except Exception as exc:
    print('CUDA cache cleanup skipped:', exc)

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_postjudge_routing_semantics_diagnostic.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_postjudge_routing_semantics_diagnostic', runner_path)
port_postjudge_routing_semantics_diagnostic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_postjudge_routing_semantics_diagnostic)

result = port_postjudge_routing_semantics_diagnostic.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'all_postjudge_routing_semantics_predictions.csv',
    'postjudge_routing_semantics_summary_by_job.csv',
    'postjudge_routing_semantics_summary_overall.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')
